# 07 · Predicción del aterrizaje con Machine Learning

**Objetivo:** entrenar y comparar cuatro clasificadores que predicen `Class` (1 = aterriza, 0 = no):

1. Regresión logística · 2. Máquina de vectores soporte (SVM) · 3. Árbol de decisión · 4. K vecinos más cercanos (KNN)

**Procedimiento (igual al laboratorio de IBM):**

* `X` = variables con One-Hot Encoding (`dataset_part_3.csv`), estandarizadas con `StandardScaler`.
* `Y` = columna `Class`.
* División entrenamiento/prueba 80/20 con `random_state=2` → 18 lanzamientos de prueba.
* Búsqueda de hiperparámetros con `GridSearchCV` y validación cruzada de 10 particiones.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
IMAGES_DIR = Path("../images")
for folder in (DATA_DIR, RESULTS_DIR, IMAGES_DIR):
    folder.mkdir(exist_ok=True)

# Repositorio público de datos del curso IBM Applied Data Science Capstone
IBM = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"


def load_csv(name: str, url: str) -> pd.DataFrame:
    """Lee data/<name> si ya existe; si no, lo descarga desde `url` y lo guarda como caché local."""
    local = DATA_DIR / name
    if local.exists():
        print(f"Leyendo {local}")
        return pd.read_csv(local)
    print(f"Descargando {url}")
    df = pd.read_csv(url)
    df.to_csv(local, index=False)
    return df

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
# Colores semánticos usados en todo el proyecto: rojo = no aterrizó, verde = aterrizó
CLASS_PALETTE = {0: "#D64545", 1: "#1FA67A"}
CLASS_LABELS = {0: "No aterrizó (0)", 1: "Aterrizó (1)"}


def save(fig, name: str) -> None:
    fig.savefig(IMAGES_DIR / name, dpi=150, bbox_inches="tight")
    print("guardada", IMAGES_DIR / name)

In [ ]:
from sklearn import preprocessing
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

data = load_csv("dataset_part_2.csv", f"{IBM}/datasets/dataset_part_2.csv")
X = load_csv("dataset_part_3.csv", f"{IBM}/datasets/dataset_part_3.csv")
print(data.shape, X.shape)

## 1. Variable objetivo y estandarización

In [ ]:
Y = data["Class"].to_numpy()
X = preprocessing.StandardScaler().fit_transform(X)
X[:2, :6]

## 2. División entrenamiento / prueba

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print("Entrenamiento:", X_train.shape[0], "| Prueba:", X_test.shape[0])

In [ ]:
def plot_confusion_matrix(y, y_predict, title: str, filename: str):
    cm = confusion_matrix(y, y_predict)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", cbar=False, annot_kws={"size": 22}, ax=ax)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Valor real")
    ax.set_title(title)
    ax.xaxis.set_ticklabels(["No aterrizó", "Aterrizó"])
    ax.yaxis.set_ticklabels(["No aterrizó", "Aterrizó"])
    save(fig, filename)
    plt.show()
    return cm


results = {}


def evaluate(name: str, search: GridSearchCV, filename: str):
    search.fit(X_train, Y_train)
    y_hat = search.predict(X_test)
    cm = plot_confusion_matrix(Y_test, y_hat, name, filename)
    results[name] = {
        "best_params": {k: (v.item() if hasattr(v, "item") else v) for k, v in search.best_params_.items()},
        "cv_accuracy": round(float(search.best_score_), 4),
        "test_accuracy": round(float(search.score(X_test, Y_test)), 4),
        "precision": round(float(precision_score(Y_test, y_hat)), 4),
        "recall": round(float(recall_score(Y_test, y_hat)), 4),
        "f1": round(float(f1_score(Y_test, y_hat)), 4),
        "confusion_matrix": cm.tolist(),
    }
    print("Mejores hiperparámetros:", search.best_params_)
    print(f"Exactitud en validación cruzada: {search.best_score_:.4f}")
    print(f"Exactitud en prueba: {results[name]['test_accuracy']:.4f}")

## 3. Regresión logística

In [ ]:
parameters = {"C": [0.01, 0.1, 1], "penalty": ["l2"], "solver": ["lbfgs"]}
logreg_cv = GridSearchCV(LogisticRegression(max_iter=1000), parameters, cv=10)
evaluate("Regresión logística", logreg_cv, "cm_logistic_regression.png")

## 4. Máquina de vectores soporte (SVM)

In [ ]:
parameters = {"kernel": ("linear", "rbf", "poly", "sigmoid"),
              "C": np.logspace(-3, 3, 5),
              "gamma": np.logspace(-3, 3, 5)}
svm_cv = GridSearchCV(SVC(), parameters, cv=10)
evaluate("SVM", svm_cv, "cm_svm.png")

## 5. Árbol de decisión

In [ ]:
parameters = {"criterion": ["gini", "entropy"],
              "splitter": ["best", "random"],
              "max_depth": [2 * n for n in range(1, 10)],
              "max_features": ["sqrt", "log2"],
              "min_samples_leaf": [1, 2, 4],
              "min_samples_split": [2, 5, 10]}
# random_state fijo para que el resultado sea reproducible (el laboratorio original no lo fija)
tree_cv = GridSearchCV(DecisionTreeClassifier(random_state=42), parameters, cv=10)
evaluate("Árbol de decisión", tree_cv, "cm_decision_tree.png")

## 6. K vecinos más cercanos (KNN)

In [ ]:
parameters = {"n_neighbors": list(range(1, 11)),
              "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
              "p": [1, 2]}
knn_cv = GridSearchCV(KNeighborsClassifier(), parameters, cv=10)
evaluate("KNN", knn_cv, "cm_knn.png")

## 7. Comparación de modelos

In [ ]:
comparison = pd.DataFrame(results).T[["cv_accuracy", "test_accuracy", "precision", "recall", "f1"]].astype(float)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
plot_df = comparison[["cv_accuracy", "test_accuracy"]].rename(
    columns={"cv_accuracy": "Validación cruzada", "test_accuracy": "Prueba"})
plot_df.plot(kind="bar", ax=ax, color=["#94A3B8", "#F26B1D"], rot=0, width=0.75)
for container in ax.containers:
    ax.bar_label(container, labels=[f"{v:.1%}" for v in container.datavalues], fontsize=12)
ax.set(ylabel="Exactitud", ylim=(0, 1.1), xlabel="")
ax.legend(loc="lower right")
save(fig, "model_accuracy.png")
plt.show()

### Selección del mejor modelo

Con solo 18 lanzamientos de prueba, cada acierto vale 5,6 puntos porcentuales; por eso varios modelos
pueden empatar en exactitud de prueba. Se desempata con la exactitud de validación cruzada (más estable,
porque promedia 10 particiones del conjunto de entrenamiento).

In [ ]:
best_name = comparison.sort_values(["test_accuracy", "cv_accuracy"], ascending=False).index[0]
print("Mejor modelo:", best_name)

summary = {
    "test_size": int(len(Y_test)),
    "train_size": int(len(Y_train)),
    "n_features": int(X.shape[1]),
    "models": results,
    "best_model": best_name,
}
(RESULTS_DIR / "model_results.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
summary["models"][best_name]

## Conclusiones

* Los cuatro modelos superan claramente la línea base y rondan el **83 %** de exactitud en prueba.
* La matriz de confusión muestra que el problema principal son los **falsos positivos**: el modelo predice
  "aterriza" en algunos lanzamientos que en realidad no aterrizaron.
* Con un dataset tan pequeño, la mejora más importante vendría de **más datos** (lanzamientos posteriores
  a 2020) y no de modelos más complejos.